# 18 — Differential abundance / composition analysis

This notebook reproduces the historical sample-level and patient-level differential-abundance/composition analysis using the final canonical annotation object from Notebook 17.

## Historical statistical semantics preserved

The historical analysis code is retained for:

- sample-level and patient-level composition tables;
- condition-level composition summaries;
- global Kruskal-Wallis tests;
- pairwise Mann-Whitney U tests;
- Benjamini-Hochberg FDR correction;
- the historical binomial-GLM sensitivity analysis;
- descriptive per-cell chi-square association tables with Cramer's V;
- stacked condition-composition plots;
- sample-level boxplots;
- the sample-level composition heatmap.

Sample-level tests remain the primary condition-comparison analysis. Per-cell chi-square tests remain descriptive only.

## Canonical input policy

This notebook requires the canonical final H5AD produced by Notebook 17:

`GBM_core_4datasets_final_annotation_CNV_consensus.h5ad`

No pre-consensus or sensitivity-analysis H5AD is accepted.

Before analysis, the notebook verifies the canonical historical CopyKAT counts, final major-compartment counts, refined-annotation count, and the historical sample/patient/condition dimensions.

## Memory-safety policy

The analysis uses only `/obs` and the number of variables. The H5AD is therefore read directly with `h5py`/`read_elem`; expression matrices and `/layers/counts` are never materialized.

This avoids the installed AnnData behavior that can allocate the complete 739,913,124-entry sparse counts layer even when `backed="r"` is requested.

## Historical GLM note

The historical executed notebook's secondary binomial-GLM sensitivity analysis produced error-status rows under its runtime. That code is intentionally retained as historical provenance and is not the primary differential-abundance test. The canonical inferential results are the sample-level non-parametric tests.


In [1]:
# =========================
# Imports and settings
# =========================
from pathlib import Path
import os
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests
import h5py

try:
    from anndata.io import read_elem
except Exception:
    from anndata._io.specs import read_elem

try:
    import statsmodels.api as sm
    HAS_STATSMODELS = True
except Exception:
    HAS_STATSMODELS = False

import matplotlib.pyplot as plt


def find_repository_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()

    for candidate in (start, *start.parents):
        if (
            (candidate / "README.md").exists()
            and (candidate / "code").exists()
            and (candidate / "metadata").exists()
        ):
            return candidate

    return start


PROJECT_DIR = Path(
    os.environ.get(
        "GLIOMA_PROJECT_ROOT",
        find_repository_root(),
    )
).expanduser().resolve()

PROCESSED_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "integrated"
)

METADATA_DIR = (
    PROJECT_DIR
    / "metadata"
)

FIGURES_DIR = (
    PROJECT_DIR
    / "figures"
    / "DA_composition_HVG8000_r0_6"
)

METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

INPUT_H5AD = (
    PROCESSED_DIR
    / "GBM_core_4datasets_final_annotation_CNV_consensus.h5ad"
)

RUN_TAG = "HVG8000_r0_6"

print(
    "INPUT_H5AD:",
    INPUT_H5AD,
)

print(
    "exists:",
    INPUT_H5AD.exists(),
)

if not INPUT_H5AD.exists():
    raise FileNotFoundError(
        "Notebook 18 requires the canonical Notebook-17 final annotation H5AD:\n"
        f"{INPUT_H5AD}"
    )


INPUT_H5AD: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_final_annotation_CNV_consensus.h5ad
exists: True


In [2]:
# =========================
# Helper functions
# =========================

def write_tsv_replace(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    df.to_csv(path, sep="	", index=False)
    print(f"[SAVED] {path} shape={df.shape}")


def savefig_replace(path, dpi=180):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close()
    print(f"[FIGURE] {path}")


def choose_col(obs, candidates, required=True):
    for c in candidates:
        if c in obs.columns:
            return c
    if required:
        raise KeyError(f"None of candidate columns found: {candidates}")
    return None


def bh_fdr(pvals):
    pvals = np.asarray(pvals, dtype=float)
    out = np.full_like(pvals, np.nan, dtype=float)
    mask = np.isfinite(pvals)
    if mask.sum() > 0:
        out[mask] = multipletests(pvals[mask], method="fdr_bh")[1]
    return out


def safe_kruskal(groups):
    valid = [np.asarray(g, dtype=float) for g in groups if len(g) >= 2 and np.nanstd(g) > 0]
    if len(valid) < 2:
        return np.nan, np.nan, "insufficient_nonconstant_groups"
    try:
        stat, p = stats.kruskal(*valid)
        return float(stat), float(p), "ok"
    except Exception as e:
        return np.nan, np.nan, f"error:{repr(e)}"


def safe_mannwhitney(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < 2 or len(y) < 2:
        return np.nan, np.nan, "insufficient_n"
    if np.nanstd(x) == 0 and np.nanstd(y) == 0 and np.nanmean(x) == np.nanmean(y):
        return np.nan, 1.0, "both_constant_equal"
    try:
        stat, p = stats.mannwhitneyu(x, y, alternative="two-sided")
        return float(stat), float(p), "ok"
    except Exception as e:
        return np.nan, np.nan, f"error:{repr(e)}"


def cramers_v_from_table(table):
    table = np.asarray(table, dtype=float)
    if table.sum() == 0:
        return np.nan
    chi2, p, dof, expected = stats.chi2_contingency(table)
    n = table.sum()
    r, k = table.shape
    denom = n * max(min(k - 1, r - 1), 1)
    return float(np.sqrt(chi2 / denom))

def assert_shape(
    df,
    expected,
    label,
):
    observed = df.shape

    if observed != expected:
        raise RuntimeError(
            f"{label} shape checkpoint failed: "
            f"{observed} != {expected}"
        )


def counts_dict(series):
    return {
        str(key): int(value)
        for key, value
        in series.value_counts(
            dropna=False
        ).items()
    }


def normalize_copykat_value(value):
    if pd.isna(value):
        return "missing"

    text = str(
        value
    ).strip().lower()

    mapping = {
        "aneuploid": "aneuploid",
        "diploid": "diploid",
        "not.defined": "not_defined",
        "not_defined": "not_defined",
        "missing": "missing",
        "nan": "missing",
    }

    return mapping.get(
        text,
        text,
    )


In [3]:
# =========================
# Load canonical final object metadata only
# =========================
if not INPUT_H5AD.exists():
    raise FileNotFoundError(
        INPUT_H5AD
    )


with h5py.File(
    INPUT_H5AD,
    "r",
) as handle:

    if "obs" not in handle:
        raise RuntimeError(
            "Canonical Notebook-17 H5AD lacks /obs."
        )

    if "var" not in handle:
        raise RuntimeError(
            "Canonical Notebook-17 H5AD lacks /var."
        )

    obs = read_elem(
        handle[
            "obs"
        ]
    )

    n_genes = int(
        read_elem(
            handle[
                "var"
            ]
        ).shape[
            0
        ]
    )


obs.index = (
    obs.index
    .astype(str)
)


print(
    "obs shape:",
    obs.shape,
)

print(
    "obs columns:",
    len(
        obs.columns
    ),
)

print(
    "n_genes:",
    n_genes,
)


condition_col = choose_col(
    obs,
    [
        "condition",
        "disease_condition",
        "group",
    ],
)

sample_col = choose_col(
    obs,
    [
        "sample_id",
        "sample",
        "orig.ident",
        "sample_name",
    ],
)

patient_col = choose_col(
    obs,
    [
        "patient_id",
        "patient",
        "donor_id",
    ],
    required=False,
)

dataset_col = choose_col(
    obs,
    [
        "core_dataset",
        "dataset_id",
        "dataset",
        "study_id",
    ],
    required=False,
)

cluster_col = choose_col(
    obs,
    [
        "leiden_scvi_main",
        "leiden_scvi_HVG8000_r0_6",
        "leiden",
    ],
)

compartment_col = choose_col(
    obs,
    [
        "final_compartment_major",
        "final_malignancy_group_copykat_marker",
    ],
)

annotation_col = choose_col(
    obs,
    [
        "final_annotation_refined",
        "annotation_marker_based",
        "annotation",
    ],
)

copykat_col = choose_col(
    obs,
    [
        "copykat_pred_clean",
        "copykat.pred",
        "copykat_pred",
    ],
    required=False,
)


print(
    "condition_col:",
    condition_col,
)

print(
    "sample_col:",
    sample_col,
)

print(
    "patient_col:",
    patient_col,
)

print(
    "dataset_col:",
    dataset_col,
)

print(
    "cluster_col:",
    cluster_col,
)

print(
    "compartment_col:",
    compartment_col,
)

print(
    "annotation_col:",
    annotation_col,
)

print(
    "copykat_col:",
    copykat_col,
)


if obs.shape[
    0
] != 351427:
    raise RuntimeError(
        "Canonical Notebook-17 cell-count checkpoint failed: "
        f"{obs.shape[0]} != 351427"
    )

if n_genes != 15176:
    raise RuntimeError(
        "Canonical Notebook-17 gene-count checkpoint failed: "
        f"{n_genes} != 15176"
    )


EXPECTED_COPYKAT_COUNTS = {
    "aneuploid": 202895,
    "diploid": 111493,
    "not_defined": 37037,
    "missing": 2,
}

observed_copykat_counts = counts_dict(
    obs[
        copykat_col
    ]
    .astype(object)
    .map(
        normalize_copykat_value
    )
)

if observed_copykat_counts != EXPECTED_COPYKAT_COUNTS:
    raise RuntimeError(
        "Notebook 18 is not consuming the exact canonical historical "
        f"CopyKAT calls. Observed: {observed_copykat_counts}"
    )


EXPECTED_COMPARTMENT_COUNTS = {
    "review": 167178,
    "malignant_like": 123250,
    "cycling_malignant_like_candidate": 38634,
    "non_malignant_or_diploid_leaning": 22365,
}

observed_compartment_counts = counts_dict(
    obs[
        compartment_col
    ]
)

if observed_compartment_counts != EXPECTED_COMPARTMENT_COUNTS:
    raise RuntimeError(
        "Notebook-17 final major-compartment checkpoint failed. "
        f"Observed: {observed_compartment_counts}"
    )


if (
    obs[
        annotation_col
    ]
    .astype(str)
    .nunique()
    != 18
):
    raise RuntimeError(
        "Notebook-17 refined-annotation checkpoint failed: expected 18 labels."
    )


print(
    "PASS: canonical Notebook-17 input checkpoints match exactly."
)


obs shape: (351427, 92)
obs columns: 92
n_genes: 15176
condition_col: condition
sample_col: sample_id
patient_col: patient_id
dataset_col: core_dataset
cluster_col: leiden_scvi_main
compartment_col: final_compartment_major
annotation_col: final_annotation_refined
copykat_col: copykat_pred_clean
PASS: canonical Notebook-17 input checkpoints match exactly.


In [4]:
# =========================
# Readiness report
# =========================
readiness_rows = []
for name, value in [
    ("n_cells", obs.shape[0]),
    ("n_genes", n_genes),
    ("condition_col", condition_col),
    ("sample_col", sample_col),
    ("patient_col", patient_col if patient_col else "NA"),
    ("dataset_col", dataset_col if dataset_col else "NA"),
    ("cluster_col", cluster_col),
    ("compartment_col", compartment_col),
    ("annotation_col", annotation_col),
    ("copykat_col", copykat_col if copykat_col else "NA"),
    ("n_conditions", obs[condition_col].astype(str).nunique()),
    ("n_samples", obs[sample_col].astype(str).nunique()),
    ("n_patients", obs[patient_col].astype(str).nunique() if patient_col else np.nan),
    ("n_clusters", obs[cluster_col].astype(str).nunique()),
    ("n_compartments", obs[compartment_col].astype(str).nunique()),
    ("n_refined_annotations", obs[annotation_col].astype(str).nunique()),
]:
    readiness_rows.append({"metric": name, "value": value})

readiness = pd.DataFrame(readiness_rows)
READINESS_TSV = METADATA_DIR / f"DA_readiness_{RUN_TAG}.tsv"
write_tsv_replace(readiness, READINESS_TSV)
display(readiness)

assert_shape(
    readiness,
    (
        16,
        2,
    ),
    "Historical DA readiness",
)

expected_readiness = {
    "n_cells": 351427,
    "n_genes": 15176,
    "n_conditions": 4,
    "n_samples": 83,
    "n_patients": 50,
    "n_clusters": 18,
    "n_compartments": 4,
    "n_refined_annotations": 18,
}

readiness_lookup = dict(
    zip(
        readiness[
            "metric"
        ],
        readiness[
            "value"
        ],
    )
)

for metric, expected in expected_readiness.items():
    observed = int(
        readiness_lookup[
            metric
        ]
    )

    if observed != expected:
        raise RuntimeError(
            f"Historical readiness checkpoint failed for {metric}: "
            f"{observed} != {expected}"
        )

print(
    "PASS: historical DA readiness checkpoints match exactly."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\DA_readiness_HVG8000_r0_6.tsv shape=(16, 2)


,metric,value
0,n_cells,351427
1,n_genes,15176
2,condition_col,condition
3,sample_col,sample_id
4,patient_col,patient_id
5,dataset_col,core_dataset
6,cluster_col,leiden_scvi_main
7,compartment_col,final_compartment_major
8,annotation_col,final_annotation_refined
9,copykat_col,copykat_pred_clean


PASS: historical DA readiness checkpoints match exactly.


In [5]:
# =========================
# Core aggregation functions
# =========================

def build_unit_feature_table(obs, unit_col, feature_col, condition_col, dataset_col=None, patient_col=None, min_unit_cells=50):
    # Return long table with count/proportion of feature categories per biological unit.
    keep_cols = [unit_col, feature_col, condition_col]
    if dataset_col and dataset_col not in keep_cols:
        keep_cols.append(dataset_col)
    if patient_col and patient_col not in keep_cols and patient_col != unit_col:
        keep_cols.append(patient_col)

    df = obs[keep_cols].copy()
    for c in keep_cols:
        df[c] = df[c].astype(str)

    unit_meta = df.groupby(unit_col).agg(
        n_cells_total=(feature_col, "size"),
        condition=(condition_col, lambda x: x.value_counts().idxmax()),
    ).reset_index()

    if dataset_col:
        unit_meta["dataset"] = df.groupby(unit_col)[dataset_col].agg(lambda x: x.value_counts().idxmax()).values
    else:
        unit_meta["dataset"] = "NA"

    if patient_col and patient_col != unit_col:
        unit_meta["patient"] = df.groupby(unit_col)[patient_col].agg(lambda x: x.value_counts().idxmax()).values
    elif unit_col == patient_col:
        unit_meta["patient"] = unit_meta[unit_col]
    else:
        unit_meta["patient"] = "NA"

    counts = df.groupby([unit_col, feature_col]).size().reset_index(name="n_cells_feature")
    out = counts.merge(unit_meta, on=unit_col, how="left")
    out = out[out["n_cells_total"] >= min_unit_cells].copy()
    out["proportion"] = out["n_cells_feature"] / out["n_cells_total"]
    out = out.rename(columns={unit_col: "unit_id", feature_col: "feature"})
    out["unit_type"] = unit_col
    out["feature_type"] = feature_col
    return out


def condition_feature_summary(unit_feature_df):
    rows = []
    for (feature, cond), sub in unit_feature_df.groupby(["feature", "condition"]):
        vals = sub["proportion"].astype(float)
        rows.append({
            "feature_type": sub["feature_type"].iloc[0],
            "feature": feature,
            "condition": cond,
            "n_units": int(vals.shape[0]),
            "mean_prop": float(vals.mean()),
            "median_prop": float(vals.median()),
            "sd_prop": float(vals.std(ddof=1)) if vals.shape[0] > 1 else 0.0,
            "min_prop": float(vals.min()),
            "max_prop": float(vals.max()),
            "total_cells_in_units": int(sub["n_cells_total"].sum()),
            "feature_cells_in_units": int(sub["n_cells_feature"].sum()),
        })
    return pd.DataFrame(rows)


def kruskal_by_feature(unit_feature_df):
    rows = []
    conditions = sorted(unit_feature_df["condition"].dropna().unique())
    for feature, sub in unit_feature_df.groupby("feature"):
        groups = [sub.loc[sub["condition"] == cond, "proportion"].values for cond in conditions]
        stat, p, status = safe_kruskal(groups)
        rows.append({
            "feature_type": sub["feature_type"].iloc[0],
            "feature": feature,
            "test": "Kruskal-Wallis",
            "conditions_compared": ";".join(conditions),
            "statistic": stat,
            "pvalue": p,
            "status": status,
            "n_units_total": int(sub["unit_id"].nunique()),
        })
    res = pd.DataFrame(rows)
    if len(res):
        res["qvalue_fdr_bh"] = bh_fdr(res["pvalue"].values)
    return res


def pairwise_mannwhitney_by_feature(unit_feature_df):
    rows = []
    conditions = sorted(unit_feature_df["condition"].dropna().unique())
    for feature, sub in unit_feature_df.groupby("feature"):
        for i, c1 in enumerate(conditions):
            for c2 in conditions[i+1:]:
                x = sub.loc[sub["condition"] == c1, "proportion"].values
                y = sub.loc[sub["condition"] == c2, "proportion"].values
                stat, p, status = safe_mannwhitney(x, y)
                rows.append({
                    "feature_type": sub["feature_type"].iloc[0],
                    "feature": feature,
                    "test": "Mann-Whitney U",
                    "condition_1": c1,
                    "condition_2": c2,
                    "n_units_1": int(len(x)),
                    "n_units_2": int(len(y)),
                    "median_prop_1": float(np.nanmedian(x)) if len(x) else np.nan,
                    "median_prop_2": float(np.nanmedian(y)) if len(y) else np.nan,
                    "mean_prop_1": float(np.nanmean(x)) if len(x) else np.nan,
                    "mean_prop_2": float(np.nanmean(y)) if len(y) else np.nan,
                    "delta_median_1_minus_2": float(np.nanmedian(x) - np.nanmedian(y)) if len(x) and len(y) else np.nan,
                    "statistic": stat,
                    "pvalue": p,
                    "status": status,
                })
    res = pd.DataFrame(rows)
    if len(res):
        res["qvalue_fdr_bh"] = bh_fdr(res["pvalue"].values)
    return res


In [6]:
# =========================
# Build sample-level tables
# =========================
MIN_SAMPLE_CELLS = 50

sample_comp = build_unit_feature_table(
    obs, sample_col, compartment_col, condition_col,
    dataset_col=dataset_col, patient_col=patient_col,
    min_unit_cells=MIN_SAMPLE_CELLS,
)
sample_ann = build_unit_feature_table(
    obs, sample_col, annotation_col, condition_col,
    dataset_col=dataset_col, patient_col=patient_col,
    min_unit_cells=MIN_SAMPLE_CELLS,
)
sample_cluster = build_unit_feature_table(
    obs, sample_col, cluster_col, condition_col,
    dataset_col=dataset_col, patient_col=patient_col,
    min_unit_cells=MIN_SAMPLE_CELLS,
)

if copykat_col:
    sample_copykat = build_unit_feature_table(
        obs, sample_col, copykat_col, condition_col,
        dataset_col=dataset_col, patient_col=patient_col,
        min_unit_cells=MIN_SAMPLE_CELLS,
    )
else:
    sample_copykat = pd.DataFrame()

SAMPLE_COMP_TSV = METADATA_DIR / f"DA_sample_level_compartment_proportions_{RUN_TAG}.tsv"
SAMPLE_ANN_TSV = METADATA_DIR / f"DA_sample_level_annotation_proportions_{RUN_TAG}.tsv"
SAMPLE_CLUSTER_TSV = METADATA_DIR / f"DA_sample_level_cluster_proportions_{RUN_TAG}.tsv"
SAMPLE_COPYKAT_TSV = METADATA_DIR / f"DA_sample_level_copykat_proportions_{RUN_TAG}.tsv"

write_tsv_replace(sample_comp, SAMPLE_COMP_TSV)
write_tsv_replace(sample_ann, SAMPLE_ANN_TSV)
write_tsv_replace(sample_cluster, SAMPLE_CLUSTER_TSV)
if copykat_col:
    write_tsv_replace(sample_copykat, SAMPLE_COPYKAT_TSV)

display(sample_comp.head())

EXPECTED_SAMPLE_SHAPES = {
    "compartment": (319, 10),
    "annotation": (1177, 10),
    "cluster": (1177, 10),
    "copykat": (249, 10),
}

observed_sample_shapes = {
    "compartment": sample_comp.shape,
    "annotation": sample_ann.shape,
    "cluster": sample_cluster.shape,
    "copykat": sample_copykat.shape,
}

if observed_sample_shapes != EXPECTED_SAMPLE_SHAPES:
    raise RuntimeError(
        "Historical sample-level DA table checkpoint failed. "
        f"Observed: {observed_sample_shapes}"
    )

print(
    "PASS: historical sample-level DA table shapes match exactly."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\DA_sample_level_compartment_proportions_HVG8000_r0_6.tsv shape=(319, 10)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\DA_sample_level_annotation_proportions_HVG8000_r0_6.tsv shape=(1177, 10)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\DA_sample_level_cluster_proportions_HVG8000_r0_6.tsv shape=(1177, 10)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\DA_sample_level_copykat_proportions_HVG8000_r0_6.tsv shape=(249, 10)


,unit_id,feature,n_cells_feature,n_cells_total,condition,dataset,patient,proportion,unit_type,feature_type
0,GSM5518596_rGBM-01-A,cycling_malignant_like_candidate,46,4324,rGBM,DS3_GSE182109,01,0.010638,sample_id,final_compartment_major
1,GSM5518596_rGBM-01-A,malignant_like,123,4324,rGBM,DS3_GSE182109,01,0.028446,sample_id,final_compartment_major
2,GSM5518596_rGBM-01-A,non_malignant_or_diploid_leaning,46,4324,rGBM,DS3_GSE182109,01,0.010638,sample_id,final_compartment_major
3,GSM5518596_rGBM-01-A,review,4109,4324,rGBM,DS3_GSE182109,01,0.950278,sample_id,final_compartment_major
4,GSM5518597_rGBM-01-B,cycling_malignant_like_candidate,144,10960,rGBM,DS3_GSE182109,01,0.013139,sample_id,final_compartment_major


PASS: historical sample-level DA table shapes match exactly.


In [7]:
# =========================
# Build patient-level tables, if patient_id exists
# =========================
MIN_PATIENT_CELLS = 50

patient_tables = {}
if patient_col:
    patient_tables["compartment"] = build_unit_feature_table(
        obs, patient_col, compartment_col, condition_col,
        dataset_col=dataset_col, patient_col=patient_col,
        min_unit_cells=MIN_PATIENT_CELLS,
    )
    patient_tables["annotation"] = build_unit_feature_table(
        obs, patient_col, annotation_col, condition_col,
        dataset_col=dataset_col, patient_col=patient_col,
        min_unit_cells=MIN_PATIENT_CELLS,
    )
    patient_tables["cluster"] = build_unit_feature_table(
        obs, patient_col, cluster_col, condition_col,
        dataset_col=dataset_col, patient_col=patient_col,
        min_unit_cells=MIN_PATIENT_CELLS,
    )
    if copykat_col:
        patient_tables["copykat"] = build_unit_feature_table(
            obs, patient_col, copykat_col, condition_col,
            dataset_col=dataset_col, patient_col=patient_col,
            min_unit_cells=MIN_PATIENT_CELLS,
        )

    for name, df in patient_tables.items():
        path = METADATA_DIR / f"DA_patient_level_{name}_proportions_{RUN_TAG}.tsv"
        write_tsv_replace(df, path)
else:
    print("[INFO] No patient column found; patient-level tables skipped.")

EXPECTED_PATIENT_SHAPES = {
    "compartment": (197, 10),
    "annotation": (703, 10),
    "cluster": (703, 10),
    "copykat": (151, 10),
}

observed_patient_shapes = {
    key: value.shape
    for key, value
    in patient_tables.items()
}

if observed_patient_shapes != EXPECTED_PATIENT_SHAPES:
    raise RuntimeError(
        "Historical patient-level DA table checkpoint failed. "
        f"Observed: {observed_patient_shapes}"
    )

print(
    "PASS: historical patient-level DA table shapes match exactly."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\DA_patient_level_compartment_proportions_HVG8000_r0_6.tsv shape=(197, 10)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\DA_patient_level_annotation_proportions_HVG8000_r0_6.tsv shape=(703, 10)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\DA_patient_level_cluster_proportions_HVG8000_r0_6.tsv shape=(703, 10)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\DA_patient_level_copykat_proportions_HVG8000_r0_6.tsv shape=(151, 10)
PASS: historical patient-level DA table shapes match exactly.


In [8]:
# =========================
# Condition summaries
# =========================
summary_comp = condition_feature_summary(sample_comp)
summary_ann = condition_feature_summary(sample_ann)
summary_cluster = condition_feature_summary(sample_cluster)
summary_copykat = condition_feature_summary(sample_copykat) if copykat_col and len(sample_copykat) else pd.DataFrame()

SUMMARY_COMP_TSV = METADATA_DIR / f"DA_condition_summary_compartment_sample_level_{RUN_TAG}.tsv"
SUMMARY_ANN_TSV = METADATA_DIR / f"DA_condition_summary_annotation_sample_level_{RUN_TAG}.tsv"
SUMMARY_CLUSTER_TSV = METADATA_DIR / f"DA_condition_summary_cluster_sample_level_{RUN_TAG}.tsv"
SUMMARY_COPYKAT_TSV = METADATA_DIR / f"DA_condition_summary_copykat_sample_level_{RUN_TAG}.tsv"

write_tsv_replace(summary_comp, SUMMARY_COMP_TSV)
write_tsv_replace(summary_ann, SUMMARY_ANN_TSV)
write_tsv_replace(summary_cluster, SUMMARY_CLUSTER_TSV)
if copykat_col and len(summary_copykat):
    write_tsv_replace(summary_copykat, SUMMARY_COPYKAT_TSV)

display(summary_comp)

EXPECTED_CONDITION_SUMMARY_SHAPES = {
    "compartment": (16, 11),
    "annotation": (72, 11),
    "cluster": (72, 11),
    "copykat": (13, 11),
}

observed_condition_summary_shapes = {
    "compartment": summary_comp.shape,
    "annotation": summary_ann.shape,
    "cluster": summary_cluster.shape,
    "copykat": summary_copykat.shape,
}

if (
    observed_condition_summary_shapes
    != EXPECTED_CONDITION_SUMMARY_SHAPES
):
    raise RuntimeError(
        "Historical condition-summary checkpoint failed. "
        f"Observed: {observed_condition_summary_shapes}"
    )

print(
    "PASS: historical condition-summary shapes match exactly."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\DA_condition_summary_compartment_sample_level_HVG8000_r0_6.tsv shape=(16, 11)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\DA_condition_summary_annotation_sample_level_HVG8000_r0_6.tsv shape=(72, 11)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\DA_condition_summary_cluster_sample_level_HVG8000_r0_6.tsv shape=(72, 11)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\DA_condition_summary_copykat_sample_level_HVG8000_r0_6.tsv shape=(13, 11)


,feature_type,feature,condition,n_units,mean_prop,median_prop,sd_prop,min_prop,max_prop,total_cells_in_units,feature_cells_in_units
0,final_compartment_major,cycling_malignant_like_candidate,GBM,39,0.127495,0.107757,0.105960,0.000603,0.346162,132692,17282
1,final_compartment_major,cycling_malignant_like_candidate,LGG,4,0.021570,0.019127,0.006817,0.016474,0.031550,13487,283
2,final_compartment_major,cycling_malignant_like_candidate,ndGBM,16,0.108287,0.073637,0.085122,0.015880,0.243145,96171,10730
3,final_compartment_major,cycling_malignant_like_candidate,rGBM,16,0.116749,0.083677,0.112934,0.001854,0.334675,90036,10339
4,final_compartment_major,malignant_like,GBM,39,0.469282,0.527081,0.264847,0.014457,0.882728,132692,65587
5,final_compartment_major,malignant_like,LGG,4,0.391904,0.339358,0.238595,0.183646,0.705254,13487,4740
6,final_compartment_major,malignant_like,ndGBM,20,0.273339,0.285899,0.220350,0.000310,0.754488,111754,34411
7,final_compartment_major,malignant_like,rGBM,17,0.239508,0.235060,0.199428,0.004119,0.598093,92064,18512
8,final_compartment_major,non_malignant_or_diploid_leaning,GBM,39,0.083415,0.037133,0.115926,0.000911,0.469667,132692,10710
9,final_compartment_major,non_malignant_or_diploid_leaning,LGG,4,0.075162,0.056369,0.054578,0.033512,0.154398,13487,1389


PASS: historical condition-summary shapes match exactly.


In [9]:
# =========================
# Global Kruskal-Wallis tests
# =========================
kw_comp = kruskal_by_feature(sample_comp)
kw_ann = kruskal_by_feature(sample_ann)
kw_cluster = kruskal_by_feature(sample_cluster)
kw_copykat = kruskal_by_feature(sample_copykat) if copykat_col and len(sample_copykat) else pd.DataFrame()

KW_COMP_TSV = METADATA_DIR / f"DA_global_kruskal_compartment_sample_level_{RUN_TAG}.tsv"
KW_ANN_TSV = METADATA_DIR / f"DA_global_kruskal_annotation_sample_level_{RUN_TAG}.tsv"
KW_CLUSTER_TSV = METADATA_DIR / f"DA_global_kruskal_cluster_sample_level_{RUN_TAG}.tsv"
KW_COPYKAT_TSV = METADATA_DIR / f"DA_global_kruskal_copykat_sample_level_{RUN_TAG}.tsv"

write_tsv_replace(kw_comp.sort_values(["qvalue_fdr_bh", "pvalue"]), KW_COMP_TSV)
write_tsv_replace(kw_ann.sort_values(["qvalue_fdr_bh", "pvalue"]), KW_ANN_TSV)
write_tsv_replace(kw_cluster.sort_values(["qvalue_fdr_bh", "pvalue"]), KW_CLUSTER_TSV)
if copykat_col and len(kw_copykat):
    write_tsv_replace(kw_copykat.sort_values(["qvalue_fdr_bh", "pvalue"]), KW_COPYKAT_TSV)

print("Compartment KW:")
display(kw_comp.sort_values(["qvalue_fdr_bh", "pvalue"]))

EXPECTED_KW_SHAPES = {
    "compartment": (4, 9),
    "annotation": (18, 9),
    "cluster": (18, 9),
    "copykat": (4, 9),
}

observed_kw_shapes = {
    "compartment": kw_comp.shape,
    "annotation": kw_ann.shape,
    "cluster": kw_cluster.shape,
    "copykat": kw_copykat.shape,
}

if observed_kw_shapes != EXPECTED_KW_SHAPES:
    raise RuntimeError(
        "Historical Kruskal-Wallis table checkpoint failed. "
        f"Observed: {observed_kw_shapes}"
    )

print(
    "PASS: historical Kruskal-Wallis table shapes match exactly."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\DA_global_kruskal_compartment_sample_level_HVG8000_r0_6.tsv shape=(4, 9)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\DA_global_kruskal_annotation_sample_level_HVG8000_r0_6.tsv shape=(18, 9)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\DA_global_kruskal_cluster_sample_level_HVG8000_r0_6.tsv shape=(18, 9)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\DA_global_kruskal_copykat_sample_level_HVG8000_r0_6.tsv shape=(4, 9)
Compartment KW:


,feature_type,feature,test,conditions_compared,statistic,pvalue,status,n_units_total,qvalue_fdr_bh
3,final_compartment_major,review,Kruskal-Wallis,GBM;LGG;ndGBM;rGBM,18.904234,0.000286,ok,83,0.001145
1,final_compartment_major,malignant_like,Kruskal-Wallis,GBM;LGG;ndGBM;rGBM,12.591980,0.005607,ok,80,0.011215
0,final_compartment_major,cycling_malignant_like_candidate,Kruskal-Wallis,GBM;LGG;ndGBM;rGBM,4.252584,0.235450,ok,75,0.313933
2,final_compartment_major,non_malignant_or_diploid_leaning,Kruskal-Wallis,GBM;LGG;ndGBM;rGBM,1.718684,0.632788,ok,81,0.632788


PASS: historical Kruskal-Wallis table shapes match exactly.


In [10]:
# =========================
# Pairwise Mann-Whitney tests
# =========================
pw_comp = pairwise_mannwhitney_by_feature(sample_comp)
pw_ann = pairwise_mannwhitney_by_feature(sample_ann)
pw_cluster = pairwise_mannwhitney_by_feature(sample_cluster)
pw_copykat = pairwise_mannwhitney_by_feature(sample_copykat) if copykat_col and len(sample_copykat) else pd.DataFrame()

PW_COMP_TSV = METADATA_DIR / f"DA_pairwise_mannwhitney_compartment_sample_level_{RUN_TAG}.tsv"
PW_ANN_TSV = METADATA_DIR / f"DA_pairwise_mannwhitney_annotation_sample_level_{RUN_TAG}.tsv"
PW_CLUSTER_TSV = METADATA_DIR / f"DA_pairwise_mannwhitney_cluster_sample_level_{RUN_TAG}.tsv"
PW_COPYKAT_TSV = METADATA_DIR / f"DA_pairwise_mannwhitney_copykat_sample_level_{RUN_TAG}.tsv"

write_tsv_replace(pw_comp.sort_values(["qvalue_fdr_bh", "pvalue"]), PW_COMP_TSV)
write_tsv_replace(pw_ann.sort_values(["qvalue_fdr_bh", "pvalue"]), PW_ANN_TSV)
write_tsv_replace(pw_cluster.sort_values(["qvalue_fdr_bh", "pvalue"]), PW_CLUSTER_TSV)
if copykat_col and len(pw_copykat):
    write_tsv_replace(pw_copykat.sort_values(["qvalue_fdr_bh", "pvalue"]), PW_COPYKAT_TSV)

print("Top pairwise compartment differences:")
display(pw_comp.sort_values(["qvalue_fdr_bh", "pvalue"]).head(20))

EXPECTED_PAIRWISE_SHAPES = {
    "compartment": (24, 16),
    "annotation": (108, 16),
    "cluster": (108, 16),
    "copykat": (24, 16),
}

observed_pairwise_shapes = {
    "compartment": pw_comp.shape,
    "annotation": pw_ann.shape,
    "cluster": pw_cluster.shape,
    "copykat": pw_copykat.shape,
}

if observed_pairwise_shapes != EXPECTED_PAIRWISE_SHAPES:
    raise RuntimeError(
        "Historical pairwise Mann-Whitney table checkpoint failed. "
        f"Observed: {observed_pairwise_shapes}"
    )

print(
    "PASS: historical pairwise Mann-Whitney table shapes match exactly."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\DA_pairwise_mannwhitney_compartment_sample_level_HVG8000_r0_6.tsv shape=(24, 16)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\DA_pairwise_mannwhitney_annotation_sample_level_HVG8000_r0_6.tsv shape=(108, 16)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\DA_pairwise_mannwhitney_cluster_sample_level_HVG8000_r0_6.tsv shape=(108, 16)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\DA_pairwise_mannwhitney_copykat_sample_level_HVG8000_r0_6.tsv shape=(24, 16)
Top pairwise compartment differences:


,feature_type,feature,test,condition_1,condition_2,n_units_1,n_units_2,median_prop_1,median_prop_2,mean_prop_1,mean_prop_2,delta_median_1_minus_2,statistic,pvalue,status,qvalue_fdr_bh
19,final_compartment_major,review,Mann-Whitney U,GBM,ndGBM,39,22,0.205074,0.627729,0.319808,0.630451,-0.422655,191.0,0.000361,ok,0.007860
20,final_compartment_major,review,Mann-Whitney U,GBM,rGBM,39,18,0.205074,0.623337,0.319808,0.616860,-0.418263,152.0,0.000655,ok,0.007860
8,final_compartment_major,malignant_like,Mann-Whitney U,GBM,rGBM,39,17,0.527081,0.235060,0.469282,0.239508,0.292021,506.0,0.001931,ok,0.015450
7,final_compartment_major,malignant_like,Mann-Whitney U,GBM,ndGBM,39,20,0.527081,0.285899,0.469282,0.273339,0.241181,555.0,0.008436,ok,0.050615
3,final_compartment_major,cycling_malignant_like_candidate,Mann-Whitney U,LGG,ndGBM,4,16,0.019127,0.073637,0.021570,0.108287,-0.054510,10.0,0.038803,ok,0.157329
0,final_compartment_major,cycling_malignant_like_candidate,Mann-Whitney U,GBM,LGG,39,4,0.107757,0.019127,0.127495,0.021570,0.088630,127.0,0.039332,ok,0.157329
18,final_compartment_major,review,Mann-Whitney U,GBM,LGG,39,4,0.205074,0.535618,0.319808,0.511364,-0.330544,43.0,0.154412,ok,0.529413
4,final_compartment_major,cycling_malignant_like_candidate,Mann-Whitney U,LGG,rGBM,4,16,0.019127,0.083677,0.021570,0.116749,-0.064550,18.0,0.211352,ok,0.634056
16,final_compartment_major,non_malignant_or_diploid_leaning,Mann-Whitney U,LGG,rGBM,4,18,0.056369,0.038651,0.075162,0.053161,0.017718,50.0,0.262201,ok,0.637744
15,final_compartment_major,non_malignant_or_diploid_leaning,Mann-Whitney U,LGG,ndGBM,4,20,0.056369,0.031184,0.075162,0.046535,0.025185,54.0,0.309053,ok,0.637744


PASS: historical pairwise Mann-Whitney table shapes match exactly.


In [11]:
# =========================
# Secondary binomial GLM sensitivity analysis
# =========================
# This is not the primary test; it is a sensitivity analysis.
# It models feature_cells / total_cells per sample as a function of condition.

def glm_binomial_condition(unit_feature_df):
    if not HAS_STATSMODELS:
        return pd.DataFrame([{"status": "statsmodels_not_available"}])

    rows = []
    for feature, sub in unit_feature_df.groupby("feature"):
        sub = sub.copy()
        if sub["condition"].nunique() < 2 or sub.shape[0] < 4:
            rows.append({"feature": feature, "status": "insufficient_conditions_or_units"})
            continue
        try:
            y = sub["proportion"].astype(float).values
            X = pd.get_dummies(sub["condition"].astype(str), prefix="condition", drop_first=True)
            X = sm.add_constant(X, has_constant="add")
            weights = sub["n_cells_total"].astype(float).values

            model = sm.GLM(y, X, family=sm.families.Binomial(), freq_weights=weights)
            fit = model.fit(maxiter=100)

            null_X = np.ones((len(y), 1))
            null = sm.GLM(y, null_X, family=sm.families.Binomial(), freq_weights=weights).fit(maxiter=100)
            lr_stat = 2 * (fit.llf - null.llf)
            df_diff = fit.df_model - null.df_model
            p = stats.chi2.sf(lr_stat, df_diff) if df_diff > 0 else np.nan

            rows.append({
                "feature_type": sub["feature_type"].iloc[0],
                "feature": feature,
                "test": "binomial_GLM_condition_LRT",
                "n_units": int(sub.shape[0]),
                "df_model": float(fit.df_model),
                "lr_statistic": float(lr_stat),
                "pvalue": float(p),
                "status": "ok",
            })
        except Exception as e:
            rows.append({"feature": feature, "test": "binomial_GLM_condition_LRT", "pvalue": np.nan, "status": f"error:{repr(e)}"})

    res = pd.DataFrame(rows)
    if "pvalue" in res.columns:
        res["qvalue_fdr_bh"] = bh_fdr(res["pvalue"].values)
    return res

glm_comp = glm_binomial_condition(sample_comp)
glm_ann = glm_binomial_condition(sample_ann)
glm_cluster = glm_binomial_condition(sample_cluster)

GLM_COMP_TSV = METADATA_DIR / f"DA_glm_binomial_compartment_sample_level_{RUN_TAG}.tsv"
GLM_ANN_TSV = METADATA_DIR / f"DA_glm_binomial_annotation_sample_level_{RUN_TAG}.tsv"
GLM_CLUSTER_TSV = METADATA_DIR / f"DA_glm_binomial_cluster_sample_level_{RUN_TAG}.tsv"

write_tsv_replace(glm_comp.sort_values(["qvalue_fdr_bh", "pvalue"], na_position="last") if "qvalue_fdr_bh" in glm_comp.columns else glm_comp, GLM_COMP_TSV)
write_tsv_replace(glm_ann.sort_values(["qvalue_fdr_bh", "pvalue"], na_position="last") if "qvalue_fdr_bh" in glm_ann.columns else glm_ann, GLM_ANN_TSV)
write_tsv_replace(glm_cluster.sort_values(["qvalue_fdr_bh", "pvalue"], na_position="last") if "qvalue_fdr_bh" in glm_cluster.columns else glm_cluster, GLM_CLUSTER_TSV)

display(glm_comp.sort_values(["qvalue_fdr_bh", "pvalue"], na_position="last") if "qvalue_fdr_bh" in glm_comp.columns else glm_comp)

EXPECTED_GLM_SHAPES = {
    "compartment": (4, 5),
    "annotation": (18, 5),
    "cluster": (18, 5),
}

observed_glm_shapes = {
    "compartment": glm_comp.shape,
    "annotation": glm_ann.shape,
    "cluster": glm_cluster.shape,
}

if observed_glm_shapes != EXPECTED_GLM_SHAPES:
    raise RuntimeError(
        "Historical GLM sensitivity-table shape checkpoint failed. "
        f"Observed: {observed_glm_shapes}"
    )

historical_glm_error_pattern = all(
    (
        "status" in table.columns
        and table[
            "status"
        ]
        .astype(str)
        .str.startswith(
            "error:"
        )
        .all()
    )
    for table in [
        glm_comp,
        glm_ann,
        glm_cluster,
    ]
)

print(
    "Historical GLM all-error runtime pattern reproduced:",
    historical_glm_error_pattern,
)

if not historical_glm_error_pattern:
    print(
        "[NOTE] The secondary GLM sensitivity analysis behaved differently "
        "under the current statsmodels/pandas runtime. The primary "
        "sample-level non-parametric tests remain the canonical analysis."
    )


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\DA_glm_binomial_compartment_sample_level_HVG8000_r0_6.tsv shape=(4, 5)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\DA_glm_binomial_annotation_sample_level_HVG8000_r0_6.tsv shape=(18, 5)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\DA_glm_binomial_cluster_sample_level_HVG8000_r0_6.tsv shape=(18, 5)


,feature,test,pvalue,status,qvalue_fdr_bh
0,cycling_malignant_like_candidate,binomial_GLM_condition_LRT,NaN,error:ValueError('Pandas data cast to numpy dt...,NaN
1,malignant_like,binomial_GLM_condition_LRT,NaN,error:ValueError('Pandas data cast to numpy dt...,NaN
2,non_malignant_or_diploid_leaning,binomial_GLM_condition_LRT,NaN,error:ValueError('Pandas data cast to numpy dt...,NaN
3,review,binomial_GLM_condition_LRT,NaN,error:ValueError('Pandas data cast to numpy dt...,NaN


Historical GLM all-error runtime pattern reproduced: True


In [12]:
# =========================
# Per-cell contingency tables for descriptive support only
# =========================
# These are not the main condition-level tests, but useful QC summaries.

cont_rows = []
for feature_col, label in [(compartment_col, "compartment"), (annotation_col, "annotation"), (cluster_col, "cluster")]:
    tab = pd.crosstab(obs[feature_col].astype(str), obs[condition_col].astype(str))
    chi2, p, dof, exp = stats.chi2_contingency(tab.values)
    cv = cramers_v_from_table(tab.values)
    cont_rows.append({
        "feature_type": label,
        "table": f"{feature_col} x {condition_col}",
        "test": "chi_square_independence_per_cell_descriptive",
        "n_cells": int(tab.values.sum()),
        "n_rows": int(tab.shape[0]),
        "n_cols": int(tab.shape[1]),
        "chi2": float(chi2),
        "dof": int(dof),
        "pvalue": float(p),
        "cramers_v": float(cv),
        "interpretation_note": "Descriptive per-cell association only; sample-level tests are primary for condition comparisons.",
    })

cont_df = pd.DataFrame(cont_rows)
CONT_TSV = METADATA_DIR / f"DA_per_cell_condition_association_descriptive_{RUN_TAG}.tsv"
write_tsv_replace(cont_df, CONT_TSV)
display(cont_df)

assert_shape(
    cont_df,
    (
        3,
        11,
    ),
    "Historical per-cell descriptive association",
)

if not (
    cont_df[
        "n_cells"
    ]
    == 351427
).all():
    raise RuntimeError(
        "Historical per-cell descriptive association cell-count checkpoint failed."
    )

print(
    "PASS: historical descriptive association table checkpoint matches."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\DA_per_cell_condition_association_descriptive_HVG8000_r0_6.tsv shape=(3, 11)


,feature_type,table,test,n_cells,n_rows,n_cols,chi2,dof,pvalue,cramers_v,interpretation_note
0,compartment,final_compartment_major x condition,chi_square_independence_per_cell_descriptive,351427,4,4,33319.618012,9,0.0,0.177775,Descriptive per-cell association only; sample-...
1,annotation,final_annotation_refined x condition,chi_square_independence_per_cell_descriptive,351427,18,4,113258.379510,51,0.0,0.327761,Descriptive per-cell association only; sample-...
2,cluster,leiden_scvi_main x condition,chi_square_independence_per_cell_descriptive,351427,18,4,113258.379510,51,0.0,0.327761,Descriptive per-cell association only; sample-...


PASS: historical descriptive association table checkpoint matches.


In [13]:
# =========================
# Figures: condition composition stacked bars
# =========================
figure_manifest = []

def plot_condition_stacked(obs, feature_col, title, filename):
    tab = pd.crosstab(obs[condition_col].astype(str), obs[feature_col].astype(str), normalize="index")
    ax = tab.plot(kind="bar", stacked=True, figsize=(10, 5), width=0.85)
    ax.set_ylabel("Fraction of cells")
    ax.set_xlabel("Condition")
    ax.set_title(title)
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    path = FIGURES_DIR / filename
    savefig_replace(path)
    figure_manifest.append({"figure": str(path), "type": "stacked_bar", "feature_col": feature_col})

plot_condition_stacked(obs, compartment_col, "Cell fraction by final compartment and condition", f"DA_condition_stacked_final_compartment_{RUN_TAG}.png")
plot_condition_stacked(obs, annotation_col, "Cell fraction by refined annotation and condition", f"DA_condition_stacked_refined_annotation_{RUN_TAG}.png")
plot_condition_stacked(obs, cluster_col, "Cell fraction by cluster and condition", f"DA_condition_stacked_cluster_{RUN_TAG}.png")
if copykat_col:
    plot_condition_stacked(obs, copykat_col, "Cell fraction by CopyKAT call and condition", f"DA_condition_stacked_copykat_{RUN_TAG}.png")


[FIGURE] G:\Repository\glioma-cnv-single-cell-atlas\figures\DA_composition_HVG8000_r0_6\DA_condition_stacked_final_compartment_HVG8000_r0_6.png
[FIGURE] G:\Repository\glioma-cnv-single-cell-atlas\figures\DA_composition_HVG8000_r0_6\DA_condition_stacked_refined_annotation_HVG8000_r0_6.png
[FIGURE] G:\Repository\glioma-cnv-single-cell-atlas\figures\DA_composition_HVG8000_r0_6\DA_condition_stacked_cluster_HVG8000_r0_6.png
[FIGURE] G:\Repository\glioma-cnv-single-cell-atlas\figures\DA_composition_HVG8000_r0_6\DA_condition_stacked_copykat_HVG8000_r0_6.png


In [14]:
# =========================
# Figures: sample-level boxplots for final compartments
# =========================

def plot_sample_box(unit_df, title, filename, max_features=12):
    feat_order = (
        unit_df.groupby("feature")["proportion"].mean()
        .sort_values(ascending=False)
        .head(max_features)
        .index.tolist()
    )
    sub = unit_df[unit_df["feature"].isin(feat_order)].copy()
    conditions = sorted(sub["condition"].unique())

    n_feat = len(feat_order)
    fig, axes = plt.subplots(n_feat, 1, figsize=(10, max(3, 2.2 * n_feat)), sharex=True)
    if n_feat == 1:
        axes = [axes]

    for ax, feat in zip(axes, feat_order):
        data = [sub.loc[(sub["feature"] == feat) & (sub["condition"] == cond), "proportion"].values for cond in conditions]
        ax.boxplot(data, labels=conditions, showfliers=False)
        for i, vals in enumerate(data, start=1):
            if len(vals):
                jitter = np.random.default_rng(42).normal(0, 0.04, size=len(vals))
                ax.scatter(np.full(len(vals), i) + jitter, vals, s=12, alpha=0.65)
        ax.set_ylabel(feat[:35])
        ax.set_ylim(bottom=0)
    axes[0].set_title(title)
    axes[-1].set_xlabel("Condition")
    path = FIGURES_DIR / filename
    savefig_replace(path)
    figure_manifest.append({"figure": str(path), "type": "sample_level_boxplot", "feature_col": unit_df["feature_type"].iloc[0]})

plot_sample_box(sample_comp, "Sample-level final compartment proportions by condition", f"DA_sample_box_final_compartment_{RUN_TAG}.png", max_features=8)
plot_sample_box(sample_cluster, "Sample-level cluster proportions by condition", f"DA_sample_box_clusters_{RUN_TAG}.png", max_features=12)


[FIGURE] G:\Repository\glioma-cnv-single-cell-atlas\figures\DA_composition_HVG8000_r0_6\DA_sample_box_final_compartment_HVG8000_r0_6.png
[FIGURE] G:\Repository\glioma-cnv-single-cell-atlas\figures\DA_composition_HVG8000_r0_6\DA_sample_box_clusters_HVG8000_r0_6.png


In [15]:
# =========================
# Figures: sample-level heatmap of final compartments
# =========================

def plot_sample_heatmap(unit_df, title, filename):
    pivot = unit_df.pivot_table(index="unit_id", columns="feature", values="proportion", fill_value=0)
    meta = unit_df[["unit_id", "condition"]].drop_duplicates().set_index("unit_id")
    if "malignant_like" in pivot.columns:
        sort_key = pivot["malignant_like"]
    else:
        sort_key = pivot.sum(axis=1)
    order = meta.assign(sort_key=sort_key).sort_values(["condition", "sort_key"], ascending=[True, False]).index
    pivot = pivot.loc[order]

    fig, ax = plt.subplots(figsize=(max(8, 0.14 * pivot.shape[0]), max(4, 0.35 * pivot.shape[1])))
    im = ax.imshow(pivot.T.values, aspect="auto", interpolation="nearest")
    ax.set_yticks(range(pivot.shape[1]))
    ax.set_yticklabels(pivot.columns, fontsize=8)
    ax.set_xticks(range(pivot.shape[0]))
    ax.set_xticklabels(pivot.index, rotation=90, fontsize=6)
    ax.set_title(title)
    ax.set_xlabel("Sample")
    ax.set_ylabel("Feature")
    cbar = fig.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label("Proportion")
    path = FIGURES_DIR / filename
    savefig_replace(path)
    figure_manifest.append({"figure": str(path), "type": "sample_level_heatmap", "feature_col": unit_df["feature_type"].iloc[0]})

plot_sample_heatmap(sample_comp, "Sample-level composition heatmap: final compartments", f"DA_sample_heatmap_final_compartment_{RUN_TAG}.png")

if len(
    figure_manifest
) != 7:
    raise RuntimeError(
        "Historical DA figure-count checkpoint failed: "
        f"{len(figure_manifest)} != 7"
    )

print(
    "PASS: historical seven-figure DA manifest count reproduced."
)


[FIGURE] G:\Repository\glioma-cnv-single-cell-atlas\figures\DA_composition_HVG8000_r0_6\DA_sample_heatmap_final_compartment_HVG8000_r0_6.png
PASS: historical seven-figure DA manifest count reproduced.


In [16]:
# =========================
# Save figure manifest and output manifest
# =========================
FIGURE_MANIFEST_TSV = METADATA_DIR / f"DA_figure_manifest_{RUN_TAG}.tsv"
write_tsv_replace(pd.DataFrame(figure_manifest), FIGURE_MANIFEST_TSV)

outputs = [
    READINESS_TSV,
    SAMPLE_COMP_TSV,
    SAMPLE_ANN_TSV,
    SAMPLE_CLUSTER_TSV,
    SUMMARY_COMP_TSV,
    SUMMARY_ANN_TSV,
    SUMMARY_CLUSTER_TSV,
    KW_COMP_TSV,
    KW_ANN_TSV,
    KW_CLUSTER_TSV,
    PW_COMP_TSV,
    PW_ANN_TSV,
    PW_CLUSTER_TSV,
    GLM_COMP_TSV,
    GLM_ANN_TSV,
    GLM_CLUSTER_TSV,
    CONT_TSV,
    FIGURE_MANIFEST_TSV,
]
if copykat_col:
    outputs.extend([SAMPLE_COPYKAT_TSV, SUMMARY_COPYKAT_TSV, KW_COPYKAT_TSV, PW_COPYKAT_TSV])

if patient_col:
    outputs.extend(list(METADATA_DIR.glob(f"DA_patient_level_*_proportions_{RUN_TAG}.tsv")))

manifest = pd.DataFrame({
    "path": [str(p) for p in outputs],
    "exists": [Path(p).exists() for p in outputs],
    "size_bytes": [Path(p).stat().st_size if Path(p).exists() else np.nan for p in outputs],
})

OUTPUT_MANIFEST_TSV = METADATA_DIR / f"DA_composition_output_manifest_{RUN_TAG}.tsv"
write_tsv_replace(manifest, OUTPUT_MANIFEST_TSV)
display(manifest)

figure_manifest_df = pd.DataFrame(
    figure_manifest
)

assert_shape(
    figure_manifest_df,
    (
        7,
        3,
    ),
    "Historical DA figure manifest",
)

assert_shape(
    manifest,
    (
        26,
        3,
    ),
    "Historical DA output manifest",
)

if not manifest[
    "exists"
].all():
    raise RuntimeError(
        "One or more expected Notebook-18 outputs are missing."
    )

print(
    "\nFINAL CHECKPOINTS"
)

print(
    "Canonical Notebook-17 input:",
    INPUT_H5AD,
)

print(
    "Cells / genes:",
    (
        obs.shape[
            0
        ],
        n_genes,
    ),
)

print(
    "Canonical CopyKAT counts:",
    EXPECTED_COPYKAT_COUNTS,
)

print(
    "Final major-compartment counts:",
    EXPECTED_COMPARTMENT_COUNTS,
)

print(
    "Readiness:",
    readiness.shape,
)

print(
    "Sample-level tables:",
    observed_sample_shapes,
)

print(
    "Patient-level tables:",
    observed_patient_shapes,
)

print(
    "Condition summaries:",
    observed_condition_summary_shapes,
)

print(
    "Kruskal-Wallis tables:",
    observed_kw_shapes,
)

print(
    "Pairwise Mann-Whitney tables:",
    observed_pairwise_shapes,
)

print(
    "GLM sensitivity tables:",
    observed_glm_shapes,
)

print(
    "Historical GLM all-error runtime pattern:",
    historical_glm_error_pattern,
)

print(
    "Per-cell descriptive association:",
    cont_df.shape,
)

print(
    "Figure manifest:",
    figure_manifest_df.shape,
)

print(
    "Output manifest:",
    manifest.shape,
)

print(
    "\n[DONE] Notebook 18 canonical differential-abundance analysis completed."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\DA_figure_manifest_HVG8000_r0_6.tsv shape=(7, 3)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\DA_composition_output_manifest_HVG8000_r0_6.tsv shape=(26, 3)


,path,exists,size_bytes
0,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,390
1,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,41280
2,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,179409
3,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,121193
4,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,2770
5,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,13897
6,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,10365
7,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,686
8,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,3137
9,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,2254



FINAL CHECKPOINTS
Canonical Notebook-17 input: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_final_annotation_CNV_consensus.h5ad
Cells / genes: (351427, 15176)
Canonical CopyKAT counts: {'aneuploid': 202895, 'diploid': 111493, 'not_defined': 37037, 'missing': 2}
Final major-compartment counts: {'review': 167178, 'malignant_like': 123250, 'cycling_malignant_like_candidate': 38634, 'non_malignant_or_diploid_leaning': 22365}
Readiness: (16, 2)
Sample-level tables: {'compartment': (319, 10), 'annotation': (1177, 10), 'cluster': (1177, 10), 'copykat': (249, 10)}
Patient-level tables: {'compartment': (197, 10), 'annotation': (703, 10), 'cluster': (703, 10), 'copykat': (151, 10)}
Condition summaries: {'compartment': (16, 11), 'annotation': (72, 11), 'cluster': (72, 11), 'copykat': (13, 11)}
Kruskal-Wallis tables: {'compartment': (4, 9), 'annotation': (18, 9), 'cluster': (18, 9), 'copykat': (4, 9)}
Pairwise Mann-Whitney tables: {'compartment': (24, 16

## Interpretation notes

- Kruskal-Wallis tests are used as the primary global test because sample-level proportions are often non-normal and sample counts are limited.
- Pairwise Mann-Whitney U tests identify which condition pairs differ after the global test.
- FDR correction is used because multiple compartments, annotations, and clusters are tested.
- Per-cell chi-square tests are included only as descriptive support, because cells from the same sample/patient are not fully independent.
- The binomial GLM is included as a secondary sensitivity analysis and should not replace sample-level nonparametric testing.
